# VIX Futures Roll Trading Strategy — Full Backtest

**Based on Buehler & Cusatis (2018)**

1. Load & clean VIX futures data (inc. pre-2008 ÷10 fix)
2. Replicate Buehler OOS (Oct 2011 – Dec 2016)
3. Train until end-2016, test 2017+ (Volmageddon & COVID out-of-sample)
4. Portfolio blend: SPX + Vol strategy with weight optimisation

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import os, glob, re, warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

try:
    import yfinance as yf
except ImportError:
    !pip install yfinance
    import yfinance as yf

print('All imports OK')

## 2. Configuration

In [ ]:
VIX_FUTURES_DIR = 'data/vix_futures'

BUEHLER_SHORT =  0.0197
BUEHLER_LONG  = -0.2400

BUEHLER_OOS_START = '2011-10-04'
BUEHLER_OOS_END   = '2016-12-30'

# Train/test split for our own optimisation
TRAIN_END   = '2016-12-30'
TEST_START  = '2017-01-03'

MONTH_CODES = {
    'F':1,'G':2,'H':3,'J':4,'K':5,'M':6,
    'N':7,'Q':8,'U':9,'V':10,'X':11,'Z':12
}

print('Config loaded.')

## 3. Load VIX Futures Data (with pre-2008 price fix)

The old CBOE archive files (2006–2007) have settle/close prices that are ~10× too high
(e.g. 130–216 instead of 13–21.6). We detect and correct this by dividing by 10
whenever a row's settle price exceeds 90 (the VIX has never closed above ~83).
This is done **per-row** rather than per-contract, because some contracts have mixed
scaling (10× early on, normal prices near expiry).

In [ ]:
def parse_expiration_from_filename(fname):
    base = os.path.basename(fname)
    m = re.match(r'VX_(\d{4}-\d{2}-\d{2})\.csv', base)
    if m:
        return pd.Timestamp(m.group(1))
    m = re.match(r'CFE_([A-Z])(\d{2})_VX\.csv', base)
    if m:
        month_code, yy = m.group(1), int(m.group(2))
        year = 2000 + yy
        month = MONTH_CODES.get(month_code)
        if month is None:
            return None
        first_day = pd.Timestamp(year, month, 1)
        wed_offset = (2 - first_day.weekday()) % 7
        third_wed = first_day + pd.Timedelta(days=wed_offset + 14)
        return third_wed
    return None


def load_single_contract(fpath):
    try:
        df = pd.read_csv(fpath)
    except Exception:
        return None
    df.columns = df.columns.str.strip()
    if df['Trade Date'].str.contains('/').any():
        df['Trade Date'] = pd.to_datetime(df['Trade Date'], format='%m/%d/%Y')
    else:
        df['Trade Date'] = pd.to_datetime(df['Trade Date'])
    df = df.rename(columns={'Trade Date': 'date'})
    expiry = parse_expiration_from_filename(fpath)
    if expiry is None:
        return None
    df['expiry'] = expiry
    
    # Use Settle, fall back to Close when Settle is 0 (early 2013 files)
    df['Settle'] = pd.to_numeric(df['Settle'], errors='coerce')
    df['Close']  = pd.to_numeric(df['Close'], errors='coerce')
    df['Settle'] = df['Settle'].where(df['Settle'] > 0, df['Close'])
    df = df[df['Settle'] > 0].copy()
    
    # ── Pre-2008 fix: divide by 10 on a PER-ROW basis ──
    # The VIX has never closed above ~83, so any individual row
    # with Settle > 90 is at 10× scale and needs dividing.
    # (Per-contract median failed because some contracts had
    #  mixed scaling: 10× early on, normal near expiry.)
    mask_scaled = df['Settle'] > 90
    df.loc[mask_scaled, 'Settle'] = df.loc[mask_scaled, 'Settle'] / 10.0
    df['_n_scaled'] = mask_scaled.sum()
    
    return df[['date', 'expiry', 'Settle', 'Total Volume', 'Open Interest', '_n_scaled']].copy()


all_files = sorted(glob.glob(os.path.join(VIX_FUTURES_DIR, '*.csv')))
print(f'Found {len(all_files)} VIX futures CSV files')

frames = []
total_rows_scaled = 0
for f in all_files:
    df = load_single_contract(f)
    if df is not None and len(df) > 0:
        total_rows_scaled += df['_n_scaled'].iloc[0]
        frames.append(df)

futures_all = pd.concat(frames, ignore_index=True)
futures_all = futures_all.sort_values(['date', 'expiry']).reset_index(drop=True)

print(f'Total rows: {len(futures_all):,}')
print(f'Date range: {futures_all.date.min().date()} to {futures_all.date.max().date()}')
print(f'Total individual rows with ÷10 fix: {total_rows_scaled}')

# Sanity check: no settle > 90 should remain
bad = futures_all[futures_all['Settle'] > 90]
if len(bad) > 0:
    print(f'⚠️ WARNING: {len(bad)} rows still have Settle > 90!')
else:
    print('✅ All settle prices ≤ 90 after fix')

## 4. Validate Pre-2008 Data After Fix

In [ ]:
# Show yearly stats to verify the fix looks sensible
print(f'{"Year":>6} {"Days":>5} {"Contracts":>10} {"F_min":>7} {"F_med":>7} {"F_max":>7}')
for yr in range(2006, 2010):
    chunk = futures_all[futures_all['date'].dt.year == yr]
    if len(chunk) == 0:
        continue
    s = chunk['Settle']
    print(f'{yr:>6} {chunk["date"].nunique():>5} {chunk["expiry"].nunique():>10} '
          f'{s.min():>7.1f} {s.median():>7.1f} {s.max():>7.1f}')

# Compare 2006-07 to 2008-09: the ranges should be broadly similar
print('\n(For reference: VIX spot ranged 10-33 in 2006-07, 16-81 in 2008-09)')

## 5. Build Daily Term Structure

In [ ]:
def build_term_structure(futures_all):
    df = futures_all[futures_all['expiry'] > futures_all['date']].copy()
    df['dte'] = (df['expiry'] - df['date']).dt.days
    records = []
    for date, group in df.groupby('date'):
        g = group.sort_values('expiry')
        if len(g) < 2:
            continue
        f1, f2 = g.iloc[0], g.iloc[1]
        records.append({
            'date': date,
            'F1_settle': f1['Settle'], 'F1_expiry': f1['expiry'], 'F1_dte': f1['dte'],
            'F2_settle': f2['Settle'], 'F2_expiry': f2['expiry'], 'F2_dte': f2['dte'],
        })
    ts = pd.DataFrame(records)
    ts['date'] = pd.to_datetime(ts['date'])
    ts = ts.set_index('date').sort_index()
    return ts

term_structure = build_term_structure(futures_all)
print(f'Term structure: {len(term_structure)} trading days')
print(f'Range: {term_structure.index.min().date()} to {term_structure.index.max().date()}')

# Check for remaining gaps
all_bdays = pd.bdate_range(term_structure.index.min(), term_structure.index.max())
missing = all_bdays.difference(term_structure.index)
print(f'Missing business days: {len(missing)}')

## 6. Download VIX Spot & ETF Prices

In [ ]:
def dl(ticker, start='2004-01-01', end='2026-12-31'):
    tmp = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    tmp = tmp[['Close']]
    if isinstance(tmp.columns, pd.MultiIndex):
        tmp.columns = tmp.columns.get_level_values(0)
    tmp.index = pd.to_datetime(tmp.index)
    if hasattr(tmp.index, 'levels'):
        tmp.index = tmp.index.get_level_values(0)
    tmp.index.name = 'date'
    return tmp

vix  = dl('^VIX').rename(columns={'Close': 'VIX'})
svxy = dl('SVXY').rename(columns={'Close': 'SVXY'})
vixy = dl('VIXY').rename(columns={'Close': 'VIXY'})
spy  = dl('SPY').rename(columns={'Close': 'SPY'})

for name, df in [('VIX', vix), ('SVXY', svxy), ('VIXY', vixy), ('SPY', spy)]:
    print(f'{name:5s}: {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}')

## 7. Merge All Data & Compute Daily Roll

In [ ]:
data = term_structure.join(vix, how='inner')
data = data.join(svxy, how='left').join(vixy, how='left').join(spy, how='left')

# Trading days to expiry
data['F1_tdte'] = data.apply(
    lambda r: max(np.busday_count(r.name.date(), r['F1_expiry'].date()), 1), axis=1
)

# Daily roll: (F1 - VIX) / trading days to settlement
# Positive in contango, negative in backwardation
data['daily_roll'] = (data['F1_settle'] - data['VIX']) / data['F1_tdte']

# ETF daily returns
data['SVXY_ret'] = data['SVXY'].pct_change()
data['VIXY_ret'] = data['VIXY'].pct_change()
data['SPY_ret']  = data['SPY'].pct_change()

# Futures return — only on same-contract days (exclude roll-day jumps)
data['F1_ret_raw'] = data['F1_settle'].pct_change()
data['F1_same_contract'] = data['F1_expiry'] == data['F1_expiry'].shift(1)
data['F1_ret'] = data['F1_ret_raw'].where(data['F1_same_contract'], 0.0)

print(f'Merged dataset: {len(data)} rows')
print(f'Range: {data.index.min().date()} to {data.index.max().date()}')

## 8. VIX Spot vs Nearest Futures & Daily Roll

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(data.index, data['VIX'], label='VIX Spot', alpha=0.7)
axes[0].plot(data.index, data['F1_settle'], label='Nearest Futures (F1)', alpha=0.7)
axes[0].set_ylabel('Level')
axes[0].legend()
axes[0].set_title('VIX Spot vs Nearest Futures Contract')

axes[1].plot(data.index, data['daily_roll'], color='steelblue', alpha=0.6, lw=0.7)
axes[1].axhline(BUEHLER_SHORT, color='green', ls='--', lw=1, label=f'Short thresh (+{BUEHLER_SHORT})')
axes[1].axhline(BUEHLER_LONG, color='red', ls='--', lw=1, label=f'Long thresh ({BUEHLER_LONG})')
axes[1].axhline(0, color='black', ls='-', lw=0.5)
axes[1].set_ylabel('Daily Roll (VIX points)')
axes[1].set_title('Daily Roll = (F1 − VIX) / Trading Days to Settlement')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Strategy Engine

In [ ]:
def generate_signals(data, short_thresh, long_thresh):
    prev_roll = data['daily_roll'].shift(1)
    signal = pd.Series('CASH', index=data.index)
    signal[prev_roll >= short_thresh] = 'SVXY'
    signal[prev_roll <= long_thresh]  = 'VIXY'
    return signal


def backtest(data, signal, label='Strategy', start=None, end=None):
    df = data.loc[start:end].copy() if start else data.copy()
    sig = signal.reindex(df.index).fillna('CASH')
    df['signal'] = sig
    df = df.dropna(subset=['SVXY_ret', 'VIXY_ret'])
    if len(df) == 0:
        return pd.DataFrame(), {}
    
    # ETF strategy returns
    df['etf_ret'] = 0.0
    df.loc[df['signal'] == 'SVXY', 'etf_ret'] = df.loc[df['signal'] == 'SVXY', 'SVXY_ret']
    df.loc[df['signal'] == 'VIXY', 'etf_ret'] = df.loc[df['signal'] == 'VIXY', 'VIXY_ret']
    
    # Futures strategy returns
    df['fut_ret'] = 0.0
    df.loc[df['signal'] == 'SVXY', 'fut_ret'] = -df.loc[df['signal'] == 'SVXY', 'F1_ret']
    df.loc[df['signal'] == 'VIXY', 'fut_ret'] = df.loc[df['signal'] == 'VIXY', 'F1_ret']
    
    # Equity curves
    df['etf_equity'] = (1 + df['etf_ret']).cumprod()
    df['fut_equity'] = (1 + df['fut_ret']).cumprod()
    df['spy_equity'] = (1 + df['SPY_ret']).cumprod()
    
    # Stats
    n = len(df); yrs = n / 252
    terminal = df['etf_equity'].iloc[-1]
    cagr = terminal ** (1/yrs) - 1 if yrs > 0 else 0
    vol = df['etf_ret'].std() * np.sqrt(252)
    sharpe = (df['etf_ret'].mean() * 252) / vol if vol > 0 else 0
    semi = df['etf_ret'][df['etf_ret'] < 0].std() * np.sqrt(252)
    sortino = (df['etf_ret'].mean() * 252) / semi if semi > 0 else 0
    dd = (df['etf_equity'] / df['etf_equity'].cummax()) - 1
    n_trades = (df['signal'] != df['signal'].shift(1)).sum()
    
    stats = {
        'Terminal ($1)': f'{terminal:.2f}',
        'CAGR': f'{cagr:.1%}', 'Vol (ann.)': f'{vol:.1%}',
        'Sharpe': f'{sharpe:.2f}', 'Sortino': f'{sortino:.2f}',
        'Max DD': f'{dd.min():.1%}',
        'Trades': n_trades,
        'Days Short': (df['signal']=='SVXY').sum(),
        'Days Long': (df['signal']=='VIXY').sum(),
        'Days Cash': (df['signal']=='CASH').sum(),
        'Period': f"{df.index.min().date()} to {df.index.max().date()}",
    }
    return df, stats

print('Engine ready.')

## 10. Buehler Replication (Oct 2011 – Dec 2016)

Using the published thresholds (+0.0197 / −0.240).

In [ ]:
sig_buehler = generate_signals(data, BUEHLER_SHORT, BUEHLER_LONG)
bt_b, stats_b = backtest(data, sig_buehler, 'Buehler ETF',
                          start=BUEHLER_OOS_START, end=BUEHLER_OOS_END)

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(bt_b.index, bt_b['etf_equity'], 'k-', lw=2, label='SVXY/VIXY Strategy')
ax.plot(bt_b.index, bt_b['fut_equity'], color='gray', lw=1.5, label='Futures Only')
ax.plot(bt_b.index, bt_b['spy_equity'], color='lightgray', lw=1.5, label='S&P 500')
ax.set_title(f'Buehler Replication ({BUEHLER_OOS_START} to {BUEHLER_OOS_END})', fontsize=14)
ax.set_ylabel('Value of $1.00 Investment')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d/%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('\n=== Buehler Replication Stats ===')
for k, v in stats_b.items():
    print(f'  {k:20s}: {v}')

## 11. Threshold Optimisation (Train: SVXY inception – Dec 2016)

Grid search optimising **Sharpe ratio** on the training set.
Everything from 2017 onward — including Volmageddon and COVID — is out-of-sample.

In [ ]:
etf_data = data.dropna(subset=['SVXY', 'VIXY']).copy()
etf_start = etf_data.index.min().strftime('%Y-%m-%d')
print(f'ETF data: {etf_start} to {etf_data.index.max().date()}')
print(f'Train: {etf_start} to {TRAIN_END}')
print(f'Test:  {TEST_START} to {etf_data.index.max().date()}')

short_grid = np.arange(0.005, 0.10, 0.005)
long_grid  = np.arange(-0.50, -0.02, 0.02)

results = []
for st in short_grid:
    for lt in long_grid:
        sig = generate_signals(data, st, lt)
        bt_tmp, stats_tmp = backtest(data, sig, '', start=etf_start, end=TRAIN_END)
        if len(bt_tmp) == 0:
            continue
        results.append({
            'short_thresh': round(st, 4),
            'long_thresh': round(lt, 4),
            'sharpe': float(stats_tmp['Sharpe']),
            'terminal': bt_tmp['etf_equity'].iloc[-1],
            'max_dd': (bt_tmp['etf_equity'] / bt_tmp['etf_equity'].cummax() - 1).min(),
        })

res_df = pd.DataFrame(results)
best = res_df.loc[res_df['sharpe'].idxmax()]
OPT_SHORT = best['short_thresh']
OPT_LONG  = best['long_thresh']

print(f'\n=== Optimal Thresholds (Train Set, max Sharpe) ===')
print(f'  Short: {OPT_SHORT:.4f}  (Buehler: {BUEHLER_SHORT})')
print(f'  Long:  {OPT_LONG:.4f}  (Buehler: {BUEHLER_LONG})')
print(f'  Sharpe: {best["sharpe"]:.3f}')
print(f'  Terminal: ${best["terminal"]:.2f}')
print(f'  Max DD: {best["max_dd"]:.1%}')

# Heatmap
pivot = res_df.pivot_table(values='sharpe', index='long_thresh', columns='short_thresh')
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.pcolormesh(pivot.columns, pivot.index, pivot.values, cmap='RdYlGn', shading='auto')
ax.plot(BUEHLER_SHORT, BUEHLER_LONG, 'k*', markersize=15, label='Buehler published')
ax.plot(OPT_SHORT, OPT_LONG, 'bs', markersize=10, label='Optimised (train)')
ax.set_xlabel('Short Threshold (contango → sell vol)')
ax.set_ylabel('Long Threshold (backwardation → buy vol)')
ax.set_title(f'Sharpe Ratio Heatmap (Train: {etf_start} to {TRAIN_END})')
ax.legend(fontsize=11)
plt.colorbar(im, label='Sharpe Ratio')
plt.tight_layout()
plt.show()

## 12. Out-of-Sample Test (2017 – Present)

Both **Volmageddon (Feb 2018)** and **COVID (Mar 2020)** are fully out-of-sample.
We compare our optimised thresholds to Buehler's original ones.

In [ ]:
test_end = etf_data.index.max().strftime('%Y-%m-%d')

sig_opt = generate_signals(data, OPT_SHORT, OPT_LONG)
bt_opt, stats_opt = backtest(data, sig_opt, 'Optimised', start=TEST_START, end=test_end)

sig_buehler_test = generate_signals(data, BUEHLER_SHORT, BUEHLER_LONG)
bt_buehler_test, stats_buehler_test = backtest(data, sig_buehler_test, 'Buehler',
                                                start=TEST_START, end=test_end)

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(bt_opt.index, bt_opt['etf_equity'], 'b-', lw=2,
        label=f'Optimised (ST={OPT_SHORT:.3f}, LT={OPT_LONG:.2f})')
ax.plot(bt_buehler_test.index, bt_buehler_test['etf_equity'], 'k--', lw=1.5,
        label='Buehler Original')
ax.plot(bt_opt.index, bt_opt['fut_equity'], color='gray', lw=1, alpha=0.7,
        label='Futures Only (opt.)')
ax.plot(bt_opt.index, bt_opt['spy_equity'], color='lightgray', lw=1.5, label='S&P 500')

for dt, lbl in [('2018-02-05', 'Volmageddon'), ('2020-03-16', 'COVID')]:
    evt = pd.Timestamp(dt)
    if evt >= bt_opt.index.min() and evt <= bt_opt.index.max():
        ax.axvline(evt, color='red', ls=':', alpha=0.5)
        ax.text(evt, ax.get_ylim()[1]*0.85, f' {lbl}', fontsize=9, color='red')

ax.set_title(f'Out-of-Sample: {TEST_START} to {test_end}\n'
             f'(Volmageddon & COVID out-of-sample)', fontsize=14)
ax.set_ylabel('Value of $1.00')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:.1f}'))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

comp = pd.DataFrame({'Optimised': stats_opt, 'Buehler Original': stats_buehler_test}).T
print('\n=== Out-of-Sample Results ===')
display(comp)

## 13. Full Period Equity Curve (Train + Test)

In [ ]:
full_start = etf_data.index.min().strftime('%Y-%m-%d')
full_end = etf_data.index.max().strftime('%Y-%m-%d')

bt_full, stats_full = backtest(data, sig_opt, 'Full', start=full_start, end=full_end)

fig, axes = plt.subplots(2, 1, figsize=(14, 10), height_ratios=[3, 1])

ax = axes[0]
ax.plot(bt_full.index, bt_full['etf_equity'], 'b-', lw=1.5, label='Vol Strategy (optimised)')
ax.plot(bt_full.index, bt_full['spy_equity'], color='gray', lw=1, label='S&P 500')
ax.axvline(pd.Timestamp(TRAIN_END), color='green', ls='--', alpha=0.5, label='Train/Test split')
for dt, lbl in [('2018-02-05', 'Volmageddon'), ('2020-03-16', 'COVID')]:
    evt = pd.Timestamp(dt)
    if evt >= bt_full.index.min():
        ax.axvline(evt, color='red', ls=':', alpha=0.4)
ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:.0f}'))
ax.set_title('Full Period: Vol Strategy vs S&P 500', fontsize=14)
ax.set_ylabel('Value of $1 (log scale)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')

ax2 = axes[1]
eq = bt_full['etf_equity']
dd = (eq / eq.cummax()) - 1
ax2.fill_between(dd.index, dd.values, 0, color='blue', alpha=0.3)
ax2.axvline(pd.Timestamp(TRAIN_END), color='green', ls='--', alpha=0.5)
ax2.set_ylabel('Drawdown')
ax2.set_title(f'Strategy Drawdown (Max: {dd.min():.1%})')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n=== Full Period Stats ===')
for k, v in stats_full.items():
    print(f'  {k:20s}: {v}')

## 14. Portfolio: S&P 500 + Vol Strategy

Blend with constant daily rebalancing. Scan 0–50% vol strategy weight
(beyond 50% is impractical for most investors).
Find weights that maximise Sharpe and minimise max drawdown.

In [ ]:
port_data = bt_full[['etf_ret', 'SPY_ret']].dropna().copy()

weights = np.arange(0.0, 0.51, 0.01)
port_results = []

for w_vol in weights:
    w_spy = 1 - w_vol
    port_ret = w_vol * port_data['etf_ret'] + w_spy * port_data['SPY_ret']
    port_eq = (1 + port_ret).cumprod()
    n_yrs = len(port_ret) / 252
    terminal = port_eq.iloc[-1]
    cagr = terminal ** (1/n_yrs) - 1
    vol = port_ret.std() * np.sqrt(252)
    sharpe = (port_ret.mean() * 252) / vol if vol > 0 else 0
    dd = (port_eq / port_eq.cummax() - 1).min()
    semi = port_ret[port_ret < 0].std() * np.sqrt(252)
    sortino = (port_ret.mean() * 252) / semi if semi > 0 else 0
    port_results.append({
        'w_vol': w_vol, 'w_spy': w_spy,
        'cagr': cagr, 'vol': vol, 'sharpe': sharpe,
        'sortino': sortino, 'max_dd': dd, 'terminal': terminal,
    })

port_df = pd.DataFrame(port_results)
best_sharpe = port_df.loc[port_df['sharpe'].idxmax()]
best_dd = port_df.loc[port_df['max_dd'].idxmax()]

print('=== Optimal Portfolio Weights ===')
print(f'\nMax Sharpe: {best_sharpe["w_vol"]:.0%} Vol + {best_sharpe["w_spy"]:.0%} SPY')
print(f'  Sharpe={best_sharpe["sharpe"]:.2f}, CAGR={best_sharpe["cagr"]:.1%}, '
      f'Max DD={best_sharpe["max_dd"]:.1%}')
print(f'\nMin Drawdown: {best_dd["w_vol"]:.0%} Vol + {best_dd["w_spy"]:.0%} SPY')
print(f'  Sharpe={best_dd["sharpe"]:.2f}, CAGR={best_dd["cagr"]:.1%}, '
      f'Max DD={best_dd["max_dd"]:.1%}')

# ── Metrics vs weight ──
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, metric, title in [
    (axes[0,0], 'sharpe', 'Sharpe Ratio'),
    (axes[0,1], 'max_dd', 'Max Drawdown'),
    (axes[1,0], 'cagr', 'CAGR'),
    (axes[1,1], 'sortino', 'Sortino Ratio'),
]:
    ax.plot(port_df['w_vol'] * 100, port_df[metric], 'b-', lw=2)
    ax.set_xlabel('Vol Strategy Weight (%)')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.axvline(10, color='orange', ls='--', alpha=0.5, label='10%')
    ax.axvline(20, color='red', ls='--', alpha=0.5, label='20%')
    ax.axvline(best_sharpe['w_vol'] * 100, color='green', ls='--', alpha=0.5, label='Max Sharpe')
    if metric in ['max_dd', 'cagr']:
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.legend(fontsize=8)

plt.suptitle('Portfolio Metrics vs Vol Strategy Allocation', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 15. Portfolio Equity Curves

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), height_ratios=[3, 1])

allocations = [
    (0.00, '100% SPY', 'lightgray'),
    (0.10, '90/10 SPY/Vol', 'green'),
    (0.20, '80/20 SPY/Vol', 'blue'),
    (best_sharpe['w_vol'],
     f'{best_sharpe["w_spy"]:.0%}/{best_sharpe["w_vol"]:.0%} SPY/Vol (max Sharpe)', 'red'),
]

ax = axes[0]
ax2 = axes[1]

for w_vol, label, color in allocations:
    port_ret = w_vol * port_data['etf_ret'] + (1 - w_vol) * port_data['SPY_ret']
    port_eq = (1 + port_ret).cumprod()
    ax.plot(port_eq.index, port_eq, color=color, lw=2 if w_vol > 0 else 1.5, label=label)
    dd = (port_eq / port_eq.cummax()) - 1
    ax2.plot(dd.index, dd, color=color, lw=0.8, alpha=0.7)

ax.axvline(pd.Timestamp(TRAIN_END), color='green', ls='--', alpha=0.3, label='Train/Test split')
for dt, lbl in [('2018-02-05', 'Volmageddon'), ('2020-03-16', 'COVID')]:
    evt = pd.Timestamp(dt)
    if evt >= port_data.index.min():
        ax.axvline(evt, color='red', ls=':', alpha=0.3)

ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:.1f}'))
ax.set_title('Portfolio Equity Curves: S&P 500 + Vol Strategy Blends', fontsize=14)
ax.set_ylabel('Value of $1 (log scale)')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3, which='both')

ax2.axvline(pd.Timestamp(TRAIN_END), color='green', ls='--', alpha=0.3)
ax2.set_ylabel('Drawdown')
ax2.set_title('Drawdowns by Allocation')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 16. Summary Table — Key Allocations

In [ ]:
summary_rows = []
for w_vol, label in [(0.0, '100% SPY'), (0.05, '95/5'), (0.10, '90/10'),
                      (0.15, '85/15'), (0.20, '80/20'),
                      (best_sharpe['w_vol'],
                       f'Max Sharpe ({best_sharpe["w_spy"]:.0%}/{best_sharpe["w_vol"]:.0%})')]:
    row = port_df.iloc[(port_df['w_vol'] - w_vol).abs().idxmin()]
    summary_rows.append({
        'Allocation': label,
        'Vol Wt': f'{row["w_vol"]:.0%}',
        'SPY Wt': f'{row["w_spy"]:.0%}',
        'CAGR': f'{row["cagr"]:.1%}',
        'Vol': f'{row["vol"]:.1%}',
        'Sharpe': f'{row["sharpe"]:.2f}',
        'Sortino': f'{row["sortino"]:.2f}',
        'Max DD': f'{row["max_dd"]:.1%}',
        'Terminal': f'${row["terminal"]:.2f}',
    })

display(pd.DataFrame(summary_rows).set_index('Allocation'))

## Notes

1. **Pre-2008 price fix**: Old CBOE archive files (2006–2007) had prices ×10. Individual rows
   with Settle > 90 are divided by 10 (per-row, not per-contract, to handle mixed scaling).

2. **2012/2013 Settle=0 gap**: Some early new-format files have Settle=0 with actual prices
   in the Close column. The loader falls back to Close automatically.

3. **SVXY leverage change**: After Volmageddon (Feb 5, 2018), SVXY changed from −1× to −0.5×.
   This is in our test set — the strategy is evaluated honestly against this structural break.

4. **Futures returns**: Roll-day jumps (when F1 switches to a new contract) are zeroed out
   to avoid spurious returns in the futures-only comparison.

5. **Transaction costs** not included (consistent with Buehler).

6. **Portfolio rebalancing**: Daily constant-weight (practical implementation would use monthly).